# mini-GPT
模仿GPT做的一个llm，只不过tokenizer是字符，每次根据前几个字符推断下一个字符。训练数据是input.txt，来自莎士比亚的作品。

https://colab.research.google.com/drive/1JMLa53HDuA-i7ZBmqV7ZnA3c_fvtXnx-?usp=sharing  
https://github.com/karpathy/ng-video-lecture/tree/master

这个文档（mini-GPT.ipynb）主要记录一些基础的模块和算法上的介绍  
主要的模型架构和训练在另外两个.py文件中： 
- 先会用bigram（一个简单的模型）来进行搭建（mini-GPT_bigram.py）
- 之后会引出transformer（mini-GPT_transformer.py） 
当然，这个transformer模型的架构与实际的transformer设计略有不同，只实现了transformer的decoder部分，encoder部分在做机器翻译的任务中会有用到。同时，这个GPT是一个预训练模型，在实际的大模型训练中，chat-GPT还会有后续的根据具体任务进行的微调。


<div style="display:flex;gap:16px;align-items:flex-start;">
  <div style="flex:1">
    <p>transfomer的架构</p>
    <img src="transformer.webp" style="width:70%">
  </div>
  <div style="flex:1">
    <p>multi-head attention</p>
    <img src="multi-head attention.jpeg" style="width:50%">
  </div>
  <div style="flex:1">
    <p>我们的transformer的架构</p>
    <img src="微信图片_20260304155815.jpg" style="width:50%">
  </div>
</div>


## 读取数据

In [2]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [3]:
# get all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [4]:
# create a mapping from characters to integers and vice versa 即字符与序号的映射
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [6]:
import torch 

# encode the entire text dataset and store it in a torch.Tensor
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

## 数据集

In [7]:
# split up the data into train and val sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

block_size = 8 # context length: how many characters do we look at when predicting the next character?

# 下面的代码仅仅是为了展示一下
x = train_data[:block_size] # 从训练数据中取出前8个字符作为输入
y = train_data[1:block_size+1] # 从训练数据中取出第2到第9个字符作为目标输出
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


## 定义怎么取batch

In [11]:
# create a function to generate a batch of data

batch_size = 4 
block_size = 8

# 关键的抽取batch的函数
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))   # 抽取batch_size个随机整数，范围是0到len(data) - block_size
    x = torch.stack([data[i:i+block_size] for i in ix])   # 对于ix中每个随机整数i，提取从i开始的block_size长度的序列，并将这些序列堆叠成一个张量x。堆叠后的x的形状是(batch_size, block_size)，其中每行是一个输入序列。
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs xb 举例:')
print(xb.shape)
print(xb)
print('targets yb 举例:')
print(yb.shape)
print(yb)

for b in range(batch_size): # batch dimension，意思是batch中第几个样本
    for t in range(block_size): # time dimension，意思是输入序列中的第几个字符
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}") # tolist()方法将PyTorch张量转换为Python列表


inputs xb 举例:
torch.Size([4, 8])
tensor([[52, 43, 47, 45, 46, 40, 53, 59],
        [39, 58,  1, 58, 46, 43,  1, 58],
        [61,  1, 47, 52,  1, 51, 63,  1],
        [ 1, 43, 51, 40, 56, 39, 41, 43]])
targets yb 举例:
torch.Size([4, 8])
tensor([[43, 47, 45, 46, 40, 53, 59, 56],
        [58,  1, 58, 46, 43,  1, 58, 47],
        [ 1, 47, 52,  1, 51, 63,  1, 40],
        [43, 51, 40, 56, 39, 41, 43, 51]])
when input is [52] the target: 43
when input is [52, 43] the target: 47
when input is [52, 43, 47] the target: 45
when input is [52, 43, 47, 45] the target: 46
when input is [52, 43, 47, 45, 46] the target: 40
when input is [52, 43, 47, 45, 46, 40] the target: 53
when input is [52, 43, 47, 45, 46, 40, 53] the target: 59
when input is [52, 43, 47, 45, 46, 40, 53, 59] the target: 56
when input is [39] the target: 58
when input is [39, 58] the target: 1
when input is [39, 58, 1] the target: 58
when input is [39, 58, 1, 58] the target: 46
when input is [39, 58, 1, 58, 46] the target: 43
when 

## 模型架构：bigram
每次只读之前的一个字符

In [12]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module): # 继承自nn.Module的语言模型类，这是一个简单的基于bigram的语言模型，即每个字符只依赖于前一个字符。

    def __init__(self, vocab_size):
        super().__init__() # 调用父类的构造函数，初始化nn.Module的一些内部机制
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size) # 定义一个嵌入层，这里的嵌入层的输入维度和输出维度都是vocab_size，这意味着每个字符索引直接映射到一个长度为vocab_size的向量，这个向量可以看作是对下一个字符的预测分布。初始时，这个嵌入层的权重是随机的。

    def forward(self, idx, targets=None): # 前向传播函数，接受输入idx和可选的targets参数。idx是一个包含当前上下文中字符索引的张量，targets是对应的目标字符索引。

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)，B是batch大小，T是时间步长，即block_size，C是词汇表大小，即嵌入维度。因为是bigram的形式，只需要通过嵌入层就能将输入的字符索引转换为对应的logits。
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C) # 将logits张量从(B, T, C)形状重塑为(B*T, C)，以便于计算交叉熵损失。每个时间步的logits被展平为一个单独的样本。另一个考虑是交叉熵损失函数期望输入的形状是(样本数量, 类别数量)，这里样本数量就是B*T，类别数量就是C，原先的形状(B, T, C)不符合这个要求，所以需要重塑为(B*T, C)。
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets) 
 
        return logits, loss



    def generate(self, idx, max_new_tokens): # 推理的时候用的函数，接受当前上下文的索引idx和要生成的新字符数量max_new_tokens。该函数通过迭代地预测下一个字符并将其添加到输入中来生成新的文本。loss是在generate之前就已经算掉了，因为loss的计算实际上就是那概率进行对比，而generate需要从概率中抽取确定的文本
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            logits, loss = self(idx) # 调用前向传播函数获取当前上下文的logits和损失。这里我们只关心logits，因为我们要根据它们来生成新的字符。这里loss没有用。这里为什么是self(idx)而不是self.forward(idx)？因为在Python中，当调用一个对象时，默认会调用它的__call__方法，而nn.Module类已经实现了__call__方法来调用forward方法，所以我们可以直接使用self(idx)来调用forward方法。
            logits = logits[:, -1, :] # becomes (B, C)，我们其实只关注最后一个时间步的logits，因为根据bigram的方法原理，我们只根据当前上下文的最后一个字符来预测下一个字符。logits[:, -1, :]表示取出logits张量中所有批次的最后一个时间步的logits，得到一个形状为(B, C)的张量
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1) 
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 2), dtype=torch.long), max_new_tokens=100)[0].tolist())) # 这里我们从一个初始的上下文开始，初始上下文是一个包含1个batch和两个时间步的张量，[[0,0]]。然后我们调用generate方法来生成100个新的字符，并将生成的索引转换回文本进行展示。生成的字符串最开始是两个\n字符，因为初始上下文是0，对应的字符是\n

torch.Size([32, 65])
tensor(4.7103, grad_fn=<NllLossBackward0>)


Sr?qP-QWktXoL&jLDJgOLVz'RIoDqHdhsV&vLLxatjscMpwLERSPyao.qfzs$Ys$zF-w,;eEkzxjgCKFChs!iWW.ObzDnxA Ms$3


### 训练

In [43]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

batch_size = 32
for steps in range(1000):

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step() # 这一步是优化器更新模型参数的步骤，optimizer.step()会根据之前计算的梯度来调整模型的参数，以最小化损失函数。这里我们使用的是AdamW优化器，它是一种自适应学习率优化算法，能够在训练过程中动态调整每个参数的学习率，从而加速收敛并提高模型性能。

print(loss.item())


2.583665132522583


In [44]:
print(decode(m.generate(idx = torch.zeros((1, 2), dtype=torch.long), max_new_tokens=100)[0].tolist()))



Thenof pre, athar wowir W:
ALI u
Bue;brgfatho rend thicr,
The trdrak,

D:
GListh R.
JO:
TERI! ce, in


## The mathematical trick in self-attention

In [ ]:
torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C) # 这里我们创建了一个形状为(B, T, C)的随机张量x，其中B是批次大小，T是时间步长，C是通道数。这个张量可以看作是一个批次中每个时间步的输入特征。
print(x.shape) # (B,T,C)

# 我们接下来要做的是识别x中的每个序列中每个元素之间的关系
# 第一种方法
xbow = torch.zeros((B,T,C)) # xbow的意思是词袋
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)，表示改时间前的所有时间步的嵌入
        xbow[b,t] = torch.mean(xprev, 0) # 对不同时间的嵌入一起取平均，(1,C)
print(xbow) # (B,T,C)

# 第二种方法
wei = torch.tril(torch.ones(T, T)) # 使用下三角矩阵，因为在计算xbow的时候，我们对于每个时间步t，只考虑了从0到t的输入字符，所以我们需要一个下三角矩阵来表示这种依赖关系。下三角矩阵中的元素为1表示对应的输入字符被包含在计算中，而上三角矩阵中的元素为0表示对应的输入字符不被包含在计算中。
wei = wei / wei.sum(1, keepdim=True) # 每一行各自求和然后归一化
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ----> (B, T, C) 。wei的第一个B维度是广播出来的
torch.allclose(xbow, xbow2) # allclose函数用于检查xbow和xbow2是否在数值上接近，返回True表示它们非常接近，说明我们使用矩阵乘法的方式计算xbow2得到了与之前逐步计算xbow相同的结果。 

torch.Size([4, 8, 2])
tensor([[[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]],

        [[ 1.3488, -0.1396],
         [ 0.8173,  0.4127],
         [-0.1342,  0.4395],
         [ 0.2711,  0.4774],
         [ 0.2421,  0.0694],
         [ 0.0084,  0.0020],
         [ 0.0712, -0.1128],
         [ 0.2527,  0.2149]],

        [[-0.6631, -0.2513],
         [ 0.1735, -0.0649],
         [ 0.1685,  0.3348],
         [-0.1621,  0.1765],
         [-0.2312, -0.0436],
         [-0.1015, -0.2855],
         [-0.2593, -0.1630],
         [-0.3015, -0.2293]],

        [[ 1.6455, -0.8030],
         [ 1.4985, -0.5395],
         [ 0.4954,  0.3420],
         [ 1.0623, -0.1802],
         [ 1.1401, -0.4462],
         [ 1.0870, -0.4071],
         [ 1.0430, -0.1299],
         [ 1.1138, -0.1641]]])


False

In [ ]:
# 第三种方法
# 引入softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf')) # masked_fill函数用于将wei张量中tril矩阵中等于0的位置填充为负无穷大（float('-inf')）。这样做的目的是在后续的softmax计算中，这些位置的权重将被置为0，其余位置的权重按时间步长进行归一化
print(wei)
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x   # (T,C)。单看每一个batch，这一步的操作是在把每一种时间长度的字符串的嵌入做一个加权平均
torch.allclose(xbow, xbow3)
wei

tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0.]])


tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [ ]:
# version 4: self-attention
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16  # head是一个注意力头
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)   # (B, T, 16)
q = query(x) # (B, T, 16)
wei =  q @ k.transpose(-2, -1) * C**-0.5 # (B, T, 16) @ (B, 16, T) ---> (B, T, T)。wei学习到了各个时间步与其他时间步之间的关系，wei[b, t, t']表示在第b个样本中，第t个时间步与第t'个时间步之间的关系强弱。这个关系是通过query和key的点积计算得到的，反映了不同时间步之间的相似性或相关性。归一化是为了防止点积的值过大导致softmax的结果过于锐化（就是倾向于变成独热）。
print(wei[0])
tril = torch.tril(torch.ones(T, T)) # 使用下三角矩阵，因为在计算xbow的时候，我们对于每个时间步t，只考虑了从0到t的输入字符，所以我们需要一个下三角矩阵来表示这种依赖关系。

wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1) # 根据时间步之间的关系进行加权平均。这能告诉你在某一时间步进行生成的时候，需要从过去整合多少的信息

v = value(x) # (B, T, 16)
out = wei @ v # (B, T, T) @ (B, T, 16) ---> (B, T, 16)

print(out[0])
out.shape

tensor([[-0.3116, -0.2300,  0.0999,  0.3821, -0.1887,  0.3470,  0.1903, -0.0801],
        [-0.5893, -0.2927,  0.0184,  0.5972, -0.3858,  0.1841, -0.0098,  0.0517],
        [-0.1808, -0.2229,  0.0135, -0.0674, -0.1740, -0.2528,  0.0132, -0.1688],
        [ 0.1385, -0.1417, -0.0595, -0.1502, -0.0990, -0.2069, -0.2285, -0.1814],
        [-0.2221,  0.0033, -0.1393, -0.2334,  0.3600,  0.1527,  0.0657,  0.1637],
        [-0.0553,  0.4270, -0.0195, -0.1755,  0.5913, -0.4460,  0.2508,  0.2156],
        [ 0.1923,  0.3474, -0.0463, -0.0558,  0.1077,  0.2230, -0.0969,  0.1423],
        [-0.3190, -0.0729, -0.1468,  0.1043, -0.1412, -0.1035,  0.1137,  0.1114]],
       grad_fn=<SelectBackward0>)
tensor([[-1.5713e-01,  8.8009e-01,  1.6152e-01, -7.8239e-01, -1.4289e-01,
          7.4676e-01,  1.0068e-01, -5.2395e-01, -8.8726e-01,  1.9068e-01,
          1.7616e-01, -5.9426e-01, -4.8124e-01, -4.8598e-01,  2.8623e-01,
          5.7099e-01],
        [ 4.1031e-01, -9.1878e-02, -1.1712e-01, -3.5821e-02, -1.

torch.Size([4, 8, 16])